---
title: "GSB5544_lab2"
format:
  html:
    theme: flatly
    embed-resources: true
execute:
  echo: true
---

# Hierarchical Data, the JSON Data Format, and APIs

In [1]:
import pandas as pd

## Shows Data

First we'll work with the "Girls" shows JSON data from the reading.

In [2]:
# Fetch data from a URL
import requests
response = requests.get("https://dlsun.github.io/pods/data/tvshows.json")

import json
data_shows = response.json()

In [3]:
df_shows = pd.json_normalize(data_shows)
# #df_shows

1\. Summarize the networks represented by these shows and the number of these shows that aired on each network.

In [4]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
print(type(data_shows), len(data_shows))
print(df_shows.shape)

print(data_shows[0].keys())
print(data_shows[0])

<class 'list'> 10
(10, 35)
dict_keys(['id', 'url', 'name', 'type', 'language', 'genres', 'status', 'runtime', 'premiered', 'officialSite', 'schedule', 'rating', 'weight', 'network', 'webChannel', 'externals', 'image', 'summary', 'updated', 'cast', 'seasons'])
{'id': 139, 'url': 'http://www.tvmaze.com/shows/139/girls', 'name': 'Girls', 'type': 'Scripted', 'language': 'English', 'genres': ['Drama', 'Romance'], 'status': 'Ended', 'runtime': 30, 'premiered': '2012-04-15', 'officialSite': 'http://www.hbo.com/girls', 'schedule': {'time': '22:00', 'days': ['Sunday']}, 'rating': {'average': 6.9}, 'weight': 75, 'network': {'id': 8, 'name': 'HBO', 'country': {'name': 'United States', 'code': 'US', 'timezone': 'America/New_York'}}, 'webChannel': None, 'externals': {'tvrage': 30124, 'thetvdb': 220411, 'imdb': 'tt1723816'}, 'image': {'medium': 'http://static.tvmaze.com/uploads/images/medium_portrait/31/78286.jpg', 'original': 'http://static.tvmaze.com/uploads/images/original_untouched/31/78286.jpg'

2\. Find the number of seasons for each show in the data. Do this two ways: one which uses `df_shows`, and another that first flattens `data_shows` to a different data frame.

In [5]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
# number of seasons for each show using df_shows
df_shows["network.name"].value_counts(dropna = False)


df_seasons = pd.json_normalize(
    data_shows,
    record_path="seasons",
    meta=["id", "name"],
    meta_prefix="show_"
)

df_seasons.head(5)

df_seasons.groupby(["show_id", "show_name"]).size().reset_index 

<bound method Series.reset_index of show_id  show_name          
139      Girls                  6
525      Gilmore Girls          8
722      The Golden Girls       7
1073     Bomb Girls             2
1955     The Powerpuff Girls    6
6771     The Powerpuff Girls    3
23542    Good Girls             3
32087    Chicken Girls          5
33320    Derry Girls            2
42726    Florida Girls          1
dtype: int64>

3\. For each episode, find the length (number of characters) of the title. Then create summaries to answer: Which show tends to have the longest episode titles? The shortest?

In [6]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
import ast

temp = df_seasons.copy()

temp["episodes"] = temp["episodes"].apply(
    lambda x: x if isinstance(x, list)
    else ast.literal_eval(x[1:-1].replace('""', '"'))
)

df_episodes = pd.json_normalize(
    temp.to_dict("records"),
    record_path="episodes",
    meta=["show_id", "show_name"]
)

df_episodes["title_length"] = df_episodes["name"].str.len()

print(df_episodes[["show_name", "name", "title_length"]])

average_lengths = df_episodes.groupby("show_name")["title_length"].mean()

print("Longest on average:")
print(average_lengths.idxmax(), average_lengths.max())

print("Shortest on average:")
print(average_lengths.idxmin(), average_lengths.min())

         show_name                          name  title_length
0            Girls                         Pilot             5
1            Girls                  Vagina Panic            12
2            Girls      All Adventurous Women Do            24
3            Girls                Hannah's Diary            14
4            Girls               Hard Being Easy            15
..             ...                           ...           ...
736  Gilmore Girls                 Hay Bale Maze            13
737  Gilmore Girls  It's Just Like Riding a Bike            28
738  Gilmore Girls             Lorelai? Lorelai?            17
739  Gilmore Girls               Unto the Breach            15
740  Gilmore Girls                    Bon Voyage            10

[741 rows x 3 columns]
Longest on average:
Gilmore Girls 21.431372549019606
Shortest on average:
Derry Girls 12.5


4\. Do any cast members in the data set share a birthday with you? Who, and what show are they on? (If there isn't anyone, try a different day.)

In [7]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
df_cast = pd.json_normalize(
    data_shows,
    record_path="cast",
    meta=["name"],
    meta_prefix="show_"
)

df_cast["birthday"] = pd.to_datetime(
    df_cast["person.birthday"],
    errors="coerce"
)

dec_26 = df_cast[
    (df_cast["birthday"].dt.month == 12) &
    (df_cast["birthday"].dt.day == 26)
]

print(dec_26[["person.name", "show_name", "person.birthday"]])

Empty DataFrame
Columns: [person.name, show_name, person.birthday]
Index: []


## TVMaze API

Now you will work with the [TVMaze API](http://www.tvmaze.com/api) from the reading. Use the API to request JSON data that you can use to answer the following questions.

1\. What was the longest show that aired in the U.S. on February 4, 2018?

_Hint:_ Use the ["Schedule" endpoint](http://www.tvmaze.com/api#schedule) to first get the data for all shows that aired on that date.

In [ ]:
# YOUR CODE HERE. ADD CELLS AS NEEDED


url = "https://api.tvmaze.com/schedule?country=US&date=2018-02-04"
response = requests.get(url)

schedule = response.json()
df_schedule = pd.json_normalize(schedule)

#df_schedule

In [9]:



longest_show = df_schedule.loc[
    df_schedule["runtime"].idxmax()
]

longest_show



id                                                                            1405375
url                                 https://www.tvmaze.com/episodes/1405375/super-...
name                                Super Bowl LII - New England Patriots vs. Phil...
season                                                                           2018
number                                                                            1.0
                                                          ...                        
show.webChannel.country.name                                                      NaN
show.webChannel.country.code                                                      NaN
show.webChannel.country.timezone                                                  NaN
show.webChannel.officialSite                                                      NaN
show.image                                                                        NaN
Name: 23, Length: 62, dtype: object

2\. Among all shows that aired in the U.S. on Feburary 4, 2018, which non-voice actors appeared on more than one show? Note: some people are credited with multiple rows on the same show and thus appear as multiple rows in the data frame.

Hint: You will need to write a for loop to make multiple requests. Use the "show.id" from the data from part 1, and use the ["Shows" endpoint](http://www.tvmaze.com/api#show-cast) to get the cast of each show. Don't forget to stagger your requests, or you will be blocked by the website!

In [ ]:
import time 

show_ids = df_schedule["show.id"].drop_duplicates().tolist()
show_names = dict(zip(df_schedule["show.id"], df_schedule["show.name"]))

cast_data = []

for show_id in show_ids:
    url = f"https://api.tvmaze.com/shows/{show_id}/cast"
    cast = requests.get(url).json()

    for person in cast:
        if person["character"].get("voice") != True:
            cast_data.append({
                "show_id": show_id,
                "show_name": show_names[show_id],
                "person_id": person["person"]["id"],
                "person_name": person["person"]["name"]
            })

    time.sleep(0.5)

    # print(cast_data)

# Tasty API

[Tasty.co](http://tasty.co) is a website and app that offers food recipes. They have made these recipes available through a [a REST API](https://rapidapi.com/apidojo/api/tasty). However, unlike the TVMaze API, this one requires authentication.

Specifically, you will need to create an account and subscribe to the "Basic" (free) plant. You will then be provided with an API key (X-Rapid-API-Key) that will need to be supplied with every request you make. This is used to track and limit usage.

1. [create an account](https://rapidapi.com/apidojo/api/tasty) and subscribe to the "Basic" (free) plan
2. log in and copy the X-RapidAPI-Key, which is a long string of letters and digits
3. paste this key to replace "PUT-YOUR-KEY-HERE" in the `headers` below

If you did everything correctly, then running the cell below should return a JSON object containing all the tags recognized by the Tasty API.

In [14]:


domain = "https://tasty.p.rapidapi.com"
endpoint = "recipes/list"
url = f"{domain}/{endpoint}"

querystring = {"from":"0","size":"20","q":"daikon"}

# TODO: Update the `headers` with your X-RapidAPI-Key.
headers = {
	"X-RapidAPI-Key": "b9ce2780f7msh755640312df97d0p1ef93ejsn257eb1147ccd",
	"X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

# Make an HTTP request to the REST API, get the JSON response.
response = requests.get(url, headers=headers, params = querystring)
# response.json()

You will need to pass these `headers` with every HTTP request to the API. The API key (X-RapidAPI-Key) is how the server keeps track of how many requests you have made.

Take a look at [the documentation](https://rapidapi.com/apidojo/api/tasty). We will use the recipes/list endpoint.

Make sure you are logged into the account you are created, and select the recipes/list endpoint from the menu at left. Notice that this brings up a form that you can fill in, which generates the corresponding code.  By default, it provides Node.js code; change this to Python with the Requests client.

1\. Search for recipes containing "daikon" (an Asian radish) and request the JSON data, and convert it to a Pandas data frame.

In [15]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

daikon_recipes = pd.json_normalize(response.json(), "results")
daikon_recipes


,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,nutrition.calories,nutrition.sugar,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,brand.image_url,brand.name,brand.id,brand.slug
0,auto,US,"[{'start_time': 7666, 'appliance': None, 'end_...","beef bulgogi, creamy and crunchy foods, easy b...",[],eng,"8757513,9295873,64455",6046,NaN,instant-pot-beef-bulgogi,...,1137.0,40.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN
1,auto,US,"[{'start_time': 9666, 'appliance': None, 'end_...",NaN,[],eng,"8757513,9295873,64461",4490,NaN,top-chef-junior-pork-banh-mi-burger,...,905.0,34.0,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN
2,auto,US,"[{'start_time': 1000, 'appliance': None, 'end_...",,[],eng,"8757513,9295873,64461",5643,NaN,banh-mi-meatball-sandwich,...,462.0,17.0,Tasty,17,under_2_hours,Under 2 hours,NaN,NaN,NaN,NaN
3,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","chicken banh mi, do chua recipe, easy weeknigh...",[],eng,"8757513,9295873,64461",9820,NaN,grilled-chicken-banh-mi,...,1026.0,22.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN
4,auto,US,"[{'start_time': 5833, 'appliance': None, 'hack...","tasty, tasty_vegetarian",[],eng,"8757513,9295873,64455",4258,NaN,how-to-make-kimchi,...,170.0,9.0,Tasty: Tasty Vegetarian,49,under_2.5_hours,Under 2.5 hours,NaN,NaN,NaN,NaN
5,auto,US,"[{'start_time': 423000, 'appliance': None, 'ha...",,[],eng,"8757513,9295873,64454",7224,NaN,japanese-omelette,...,412.0,8.0,Tasty 101,63,under_15_minutes,Under 15 minutes,NaN,NaN,NaN,NaN
6,auto,US,"[{'start_time': 8000, 'appliance': None, 'end_...","chicken, dairy-free, dinner, gluten-free, heal...",[],eng,"8757513,9295873,64460",4621,NaN,low-carb-pad-thai,...,310.0,10.0,Goodful,34,under_45_minutes,Under 45 minutes,NaN,NaN,NaN,NaN
7,auto,US,"[{'start_time': 0, 'appliance': None, 'hacks':...","breakfast, chinese, hawker food, international...",[],eng,"9295813,64486,64459,9299649",6542,NaN,fried-carrot-cake,...,196.0,2.0,Tasty,17,NaN,NaN,https://img.buzzfeed.com/tasty-app-user-assets...,VisitSingapore,24.0,visitsingapore
8,auto,US,"[{'start_time': 7166, 'appliance': None, 'end_...","beef, beef bulgogi, bibimbap, bulgogi, buzzfee...",[],eng,"8757513,9295873,64455",4013,NaN,bibimbap-by-chef-esther-choi,...,1382.0,13.0,Tasty,17,under_1.5_hours,Under 1.5 hours,NaN,NaN,NaN,NaN
9,auto,US,"[{'start_time': 60000, 'appliance': None, 'end...","banh mi recipe, daikon, grilled lemongrass por...",[],eng,"8757513,9295873,64461",5152,NaN,grilled-lemongrass-pork-banh-mi,...,1224.0,26.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN


2\. How many recipes containing daikon are there? Which one is the cheapest per portion?

In [16]:
from sqlalchemy.engine import result
# YOUR CODE HERE. ADD CELLS AS NEEDED
# how many recipes
len(daikon_recipes)
# cheapest per portion

cheapest = daikon_recipes["cost_per_portion"] = (
    daikon_recipes["price.total"] /
    daikon_recipes["num_servings"]
)
cheapest = daikon_recipes.sort_values(
    "cost_per_portion",
    ascending=True
)
cheapest

,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,nutrition.sugar,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,brand.image_url,brand.name,brand.id,brand.slug,cost_per_portion
10,auto,US,"[{'start_time': 5500, 'appliance': None, 'end_...","chinese cuisine, easy tofu recipe, hoisin sauc...",[],eng,"8757513,9295873,64448",4711,NaN,vegan-tofu-bao-buns-with-pickled-vegetables,...,21.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN,304.166667
6,auto,US,"[{'start_time': 8000, 'appliance': None, 'end_...","chicken, dairy-free, dinner, gluten-free, heal...",[],eng,"8757513,9295873,64460",4621,NaN,low-carb-pad-thai,...,10.0,Goodful,34,under_45_minutes,Under 45 minutes,NaN,NaN,NaN,NaN,500.000000
1,auto,US,"[{'start_time': 9666, 'appliance': None, 'end_...",NaN,[],eng,"8757513,9295873,64461",4490,NaN,top-chef-junior-pork-banh-mi-burger,...,34.0,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN,650.000000
4,auto,US,"[{'start_time': 5833, 'appliance': None, 'hack...","tasty, tasty_vegetarian",[],eng,"8757513,9295873,64455",4258,NaN,how-to-make-kimchi,...,9.0,Tasty: Tasty Vegetarian,49,under_2.5_hours,Under 2.5 hours,NaN,NaN,NaN,NaN,650.000000
2,auto,US,"[{'start_time': 1000, 'appliance': None, 'end_...",,[],eng,"8757513,9295873,64461",5643,NaN,banh-mi-meatball-sandwich,...,17.0,Tasty,17,under_2_hours,Under 2 hours,NaN,NaN,NaN,NaN,675.000000
3,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","chicken banh mi, do chua recipe, easy weeknigh...",[],eng,"8757513,9295873,64461",9820,NaN,grilled-chicken-banh-mi,...,22.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN,812.500000
9,auto,US,"[{'start_time': 60000, 'appliance': None, 'end...","banh mi recipe, daikon, grilled lemongrass por...",[],eng,"8757513,9295873,64461",5152,NaN,grilled-lemongrass-pork-banh-mi,...,26.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN,850.000000
7,auto,US,"[{'start_time': 0, 'appliance': None, 'hacks':...","breakfast, chinese, hawker food, international...",[],eng,"9295813,64486,64459,9299649",6542,NaN,fried-carrot-cake,...,2.0,Tasty,17,NaN,NaN,https://img.buzzfeed.com/tasty-app-user-assets...,VisitSingapore,24.0,visitsingapore,868.750000
11,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","tasty, tasty_vegetarian",[],eng,"8757513,9295873,64461",4326,NaN,how-to-make-vegan-pho,...,10.0,Tasty: Tasty Vegetarian,49,NaN,NaN,NaN,NaN,NaN,NaN,900.000000
14,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...","banh mi, burnt end bánh mì, sandwich, vietnamese",[],eng,"8757513,9295873,64461",9225,NaN,burnt-end-banh-mi,...,53.0,Tasty,17,NaN,NaN,NaN,NaN,NaN,NaN,1025.000000


3\. Find recipes containing avocado and request the JSON data.

**Hint:** Note that there are hundreds of results, but the API only returns 20 results by default and only 40 results maximum, even if you specify the `size=` parameter, so you will need to use a `for` loop, incrementing the `from=` parameter. Be sure to respect the API's rate limits (or you may be blocked!)

**Note:** Recall from the example in the reading that we created an empty list `episodes = []` and then added the results of each request to it in the for loop with `episodes.extend(response.json())`. But note that in the recipes/list endpoint there are two keys: count and results. We only want the results so try instead `.extend(response.json()).get("results")`.

**Suggestion:** Try writing a loop that only makes 2 or 3 requests first so you can test that it's working correctly. Also, make sure you use separate cells for code that makes requests to the API versus processing of the results; you don't want to rerun the requests unless absolutely necessary!

In [17]:
# YOUR CODE HERE. ADD CELLS AS NEEDED

avocado_recipes = []

# First request
querystring = {
    "q": "avocado",
    "from": 0,
    "size": 40
}

response = requests.get(
    url,
    headers=headers,
    params=querystring
)

response.raise_for_status()
data = response.json()

total_recipes = data["count"]
avocado_recipes.extend(data["results"])

# Remaining requests
for start in range(40, total_recipes, 40):
    querystring = {
        "q": "avocado",
        "from": start,
        "size": 40
    }

    response = requests.get(
        url,
        headers=headers,
        params=querystring
    )

    response.raise_for_status()
    avocado_recipes.extend(response.json()["results"])

    time.sleep(1)

4\. For the recipes containing avocado, compute the proportion of reviews that are positive. For the avocado recipes with over 500 reviews, which one has the highest proportion of positive reviews?


In [19]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
avocado_recipes = pd.json_normalize(avocado_recipes)
avocado_recipes["total_reviews"] = (
    avocado_recipes["user_ratings.count_positive"] +
    avocado_recipes["user_ratings.count_negative"]
)

avocado_recipes["positive_proportion"] = (
    avocado_recipes["user_ratings.count_positive"] /
    avocado_recipes["total_reviews"]
)



In [20]:
over_500 = avocado_recipes[
    avocado_recipes["total_reviews"] > 500
]
highest = over_500.sort_values(
    "positive_proportion",
    ascending=False
).iloc[0]

print(highest[["name", "positive_proportion", "total_reviews"]])

name                   Grilled Salmon With Avocado Salsa
positive_proportion                             0.987355
total_reviews                                     1898.0
Name: 14, dtype: object


5\. Take the avocado JSON data from above (you do NOT need to read in the data from the REST API again). How many recipes are vegetarian? You should be able to identify this from the "tags" attribute.

Hint: Try using `json_normalize` with "tags" as the record path to flatten the data so that there is one row for each tag.

In [21]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
df_tags = pd.json_normalize(
    avocado_recipes.to_dict("records"),
    record_path="tags",
    meta=["id", "name"],
    meta_prefix="recipe_"
)
vegetarian = df_tags[
    df_tags["display_name"].str.lower() == "vegetarian"
]

vegetarian["id"].nunique()

vegetarian = df_tags[
    df_tags["name"].str.lower() == "vegetarian"
]

len(vegetarian)

212